In [1]:
# Install MediaPipe
!pip install -q mediapipe

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Imports
import os
import cv2
import joblib
import numpy as np
import mediapipe as mp

from base64 import b64decode
from collections import deque, Counter
from IPython.display import display, Javascript
from google.colab import output

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

PROJECT_ROOT = "/content/drive/MyDrive/NSL_Project"

MODEL_FILE = os.path.join(
    PROJECT_ROOT,
    "models",
    "nsl_classifier_v4_geometry.pkl"
)

classifier = joblib.load(MODEL_FILE)

CONFIDENCE_THRESHOLD = 0.50

print("✅ Setup complete")
print("Model:", MODEL_FILE)
print("Features:", classifier.n_features_in_)
print("Classes:", classifier.classes_)
print("Threshold:", CONFIDENCE_THRESHOLD)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete
Model: /content/drive/MyDrive/NSL_Project/models/nsl_classifier_v4_geometry.pkl
Features: 28
Classes: ['Gha' 'Ka' 'Sa' 'anga' 'cha' 'dha' 'gya' 'ha' 'la' 'pa' 'ra']
Threshold: 0.5


In [2]:
HAND_MODEL = "/content/hand_landmarker.task"

# Download model into the current Colab runtime
!wget -q \
https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task \
-O /content/hand_landmarker.task

# Make sure it actually exists
if not os.path.exists(HAND_MODEL):
    raise FileNotFoundError(
        "MediaPipe hand model failed to download."
    )

base_options = python.BaseOptions(
    model_asset_path=HAND_MODEL
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(
    options
)

print("✅ MediaPipe hand detector ready")
print("Model file:", HAND_MODEL)

✅ MediaPipe hand detector ready
Model file: /content/hand_landmarker.task


In [3]:
def normalize_landmarks(hand_landmarks):

    landmarks = np.array(
        [[lm.x, lm.y, lm.z] for lm in hand_landmarks],
        dtype=np.float32
    )

    # Move wrist to origin
    wrist = landmarks[0].copy()
    landmarks = landmarks - wrist

    # Normalize hand size
    distances = np.linalg.norm(
        landmarks[:, :2],
        axis=1
    )

    scale = np.max(distances)

    if scale > 0:
        landmarks = landmarks / scale

    return landmarks.flatten()

ANGLE_TRIPLETS = [
    (0,1,2), (1,2,3), (2,3,4),

    (0,5,6), (5,6,7), (6,7,8),

    (0,9,10), (9,10,11), (10,11,12),

    (0,13,14), (13,14,15), (14,15,16),

    (0,17,18), (17,18,19), (18,19,20),

    (5,0,9),
    (9,0,13),
    (13,0,17)
]

def angle_3d(a, b, c):

    ba = a - b
    bc = c - b

    denominator = (
        np.linalg.norm(ba) *
        np.linalg.norm(bc)
    )

    if denominator < 1e-8:
        return 0.0

    cosine = np.dot(ba, bc) / denominator
    cosine = np.clip(cosine, -1.0, 1.0)

    return np.arccos(cosine) / np.pi

def make_geometry_only_features(flat_landmarks):

    points = np.array(
        flat_landmarks,
        dtype=np.float32
    ).reshape(21, 3)

    features = []

    # 18 joint-angle features
    for a, b, c in ANGLE_TRIPLETS:

        features.append(
            angle_3d(
                points[a],
                points[b],
                points[c]
            )
        )

    # 10 distances between fingertips
    fingertips = [4, 8, 12, 16, 20]

    for i in range(len(fingertips)):

        for j in range(i + 1, len(fingertips)):

            distance = np.linalg.norm(
                points[fingertips[i]] -
                points[fingertips[j]]
            )

            features.append(distance)

    return np.array(
        features,
        dtype=np.float32
    )

# Safety check
dummy = np.zeros(63, dtype=np.float32)

print(
    "V4 feature count:",
    len(make_geometry_only_features(dummy))
)

DISPLAY_NAMES = {
    "Ka":   ("क", "Ka"),
    "Gha":  ("घ", "Gha"),
    "Sa":   ("स", "Sa"),
    "anga": ("अं", "Anga"),
    "dha":  ("ध", "Dha"),
    "ha":   ("ह", "Ha"),
    "la":   ("ल", "La"),
    "pa":   ("प", "Pa"),
    "ra":   ("र", "Ra"),
    "cha":  ("च", "Cha"),
    "gya":  ("ज्ञ", "Gya")
}

# 4. TEMPORAL SMOOTHING

prediction_history = deque(maxlen=7)

def process_nsl_frame_builder(image_data):

    global prediction_history

    try:

        # Decode browser JPEG
        encoded = image_data.split(",")[1]
        binary = b64decode(encoded)

        image_array = np.frombuffer(
            binary,
            dtype=np.uint8
        )

        frame = cv2.imdecode(
            image_array,
            cv2.IMREAD_COLOR
        )

        # Convert BGR -> RGB
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        result = detector.detect(mp_image)

        if not result.hand_landmarks:

            prediction_history.clear()

            return {
                "label": "No hand",
                "nepali": "",
                "confidence": 0
            }

        landmarks_63 = normalize_landmarks(
            result.hand_landmarks[0]
        )

        features_28 = make_geometry_only_features(
            landmarks_63
        )

        probabilities = classifier.predict_proba(
            [features_28]
        )[0]

        best_index = np.argmax(
            probabilities
        )

        raw_prediction = str(
            classifier.classes_[best_index]
        )

        confidence = float(
            probabilities[best_index]
        )

        # UNKNOWN REJECTION

        if confidence < CONFIDENCE_THRESHOLD:
            raw_prediction = "Unknown"

        # TEMPORAL SMOOTHING

        prediction_history.append(
            raw_prediction
        )

        # Need at least 5 frames first
        if len(prediction_history) < 5:

            return {
                "label": "Reading...",
                "nepali": "",
                "confidence": confidence
            }

        counts = Counter(
            prediction_history
        )

        stable_label, count = (
            counts.most_common(1)[0]
        )

        # Require 5 matching results
        if count < 5:

            return {
                "label": "Reading...",
                "nepali": "",
                "confidence": confidence
            }

        if stable_label == "Unknown":

            return {
                "label": "Unknown",
                "nepali": "",
                "confidence": confidence
            }

        nepali, english = DISPLAY_NAMES.get(
            stable_label,
            ("", stable_label)
        )

        return {
            "label": english,
            "nepali": nepali,
            "confidence": confidence
        }

    except Exception as e:

        return {
            "label": "ERROR",
            "nepali": "",
            "confidence": 0,
            "error": str(e)
        }

output.register_callback(
    "nsl.processFrameBuilder",
    process_nsl_frame_builder
)

print(" V4 recognition functions ready")
print("Word-builder callback registered")
print("Threshold:", CONFIDENCE_THRESHOLD)

V4 feature count: 28
 V4 recognition functions ready
Word-builder callback registered
Threshold: 0.5


In [8]:
display(Javascript(r"""
(async () => {

    // Stop an old camera if it is still running
    if (window.nslStream) {
        window.nslStream.getTracks().forEach(track => track.stop());
    }

    const oldApp = document.getElementById("nsl-app");

    if (oldApp) {
        oldApp.remove();
    }

    // Main container
    const container = document.createElement("div");
    container.id = "nsl-app";

    container.style.cssText = `
        width: 640px;
        font-family: Arial, sans-serif;
        text-align: center;
    `;

    // Title
    const title = document.createElement("h2");
    title.innerText = "Sign Language Recognizer";

    // Camera
    const video = document.createElement("video");

    video.width = 640;
    video.height = 480;
    video.autoplay = true;
    video.playsInline = true;

    video.style.cssText = `
        width: 640px;
        height: 480px;
        background: black;
        transform: scaleX(-1);
    `;

    // Nepali character
    const nepaliCharacter = document.createElement("div");

    nepaliCharacter.style.cssText = `
        font-size: 55px;
        font-weight: bold;
        margin-top: 10px;
        min-height: 65px;
    `;

    // English label
    const prediction = document.createElement("div");

    prediction.style.cssText = `
        font-size: 24px;
        font-weight: bold;
    `;

    prediction.innerText = "Starting camera...";

    // Confidence
    const confidence = document.createElement("div");

    confidence.style.cssText = `
        font-size: 17px;
        margin: 5px;
        min-height: 22px;
    `;

    // Text heading
    const textLabel = document.createElement("div");

    textLabel.innerText = "Built Text:";

    textLabel.style.cssText = `
        margin-top: 15px;
        font-size: 16px;
        text-align: left;
    `;

    // Text box
    const builtText = document.createElement("div");

    builtText.style.cssText = `
        min-height: 45px;
        border: 1px solid #999;
        padding: 10px;
        margin-top: 5px;
        margin-bottom: 10px;
        font-size: 30px;
        text-align: left;
        background: white;
        color: black;
    `;

    let text = "";
    let currentCharacter = "";
    let running = true;

    // Button helper
    function makeButton(label) {

        const button = document.createElement("button");

        button.innerText = label;

        button.style.cssText = `
            padding: 8px 15px;
            margin: 4px;
            cursor: pointer;
        `;

        return button;
    }

    const addButton = makeButton("Add");
    const spaceButton = makeButton("Space");
    const deleteButton = makeButton("Delete");
    const clearButton = makeButton("Clear");
    const stopButton = makeButton("Stop Camera");

    // Add current recognized character
    addButton.onclick = () => {

        if (currentCharacter !== "") {

            text += currentCharacter;
            builtText.innerText = text;
        }
    };

    // Add a space
    spaceButton.onclick = () => {

        text += " ";
        builtText.innerText = text;
    };

    // Delete last character
    deleteButton.onclick = () => {

        const characters = Array.from(text);

        characters.pop();

        text = characters.join("");

        builtText.innerText = text;
    };

    // Clear text
    clearButton.onclick = () => {

        text = "";
        builtText.innerText = "";
    };

    // Stop camera
    stopButton.onclick = () => {

        running = false;

        if (window.nslStream) {

            window.nslStream
                .getTracks()
                .forEach(track => track.stop());
        }

        prediction.innerText = "Camera stopped";
        confidence.innerText = "";
        nepaliCharacter.innerText = "";
        currentCharacter = "";
    };

    // Add elements to page
    container.appendChild(title);
    container.appendChild(video);

    container.appendChild(nepaliCharacter);
    container.appendChild(prediction);
    container.appendChild(confidence);

    container.appendChild(textLabel);
    container.appendChild(builtText);

    container.appendChild(addButton);
    container.appendChild(spaceButton);
    container.appendChild(deleteButton);
    container.appendChild(clearButton);

    container.appendChild(
        document.createElement("br")
    );

    container.appendChild(stopButton);

    document.body.appendChild(container);

    // Start webcam
    const stream =
        await navigator.mediaDevices.getUserMedia({
            video: {
                width: 640,
                height: 480
            }
        });

    window.nslStream = stream;

    video.srcObject = stream;

    await video.play();

    // Canvas used to send smaller frames to Python
    const canvas = document.createElement("canvas");

    canvas.width = 480;
    canvas.height = 360;

    const context = canvas.getContext("2d");

    // Process camera frames
    async function processFrame() {

        if (!running) {
            return;
        }

        context.drawImage(
            video,
            0,
            0,
            canvas.width,
            canvas.height
        );

        const imageData =
            canvas.toDataURL(
                "image/jpeg",
                0.70
            );

        try {

            const result =
                await google.colab.kernel.invokeFunction(
                    "nsl.processFrameBuilder",
                    [imageData],
                    {}
                );

            let resultText =
                result.data["text/plain"];

            // Convert Python dictionary text into JSON
            resultText = resultText
                .replace(/'/g, '"')
                .replace(/None/g, 'null')
                .replace(/True/g, 'true')
                .replace(/False/g, 'false');

            const data = JSON.parse(resultText);

            prediction.innerText = data.label;

            // Stable known sign
            if (data.nepali) {

                nepaliCharacter.innerText =
                    data.nepali;

                currentCharacter =
                    data.nepali;
            }

            else {

                nepaliCharacter.innerText = "";
                currentCharacter = "";
            }

            // Confidence
            if (
                data.label === "No hand" ||
                data.label === "Reading..."
            ) {

                confidence.innerText = "";
            }

            else {

                confidence.innerText =
                    "Confidence: " +
                    Math.round(
                        data.confidence * 100
                    ) +
                    "%";
            }

            if (data.label === "ERROR") {

                confidence.innerText =
                    data.error || "Unknown error";

                currentCharacter = "";
            }

        }

        catch (error) {

            prediction.innerText = "Error";

            confidence.innerText =
                error.toString();

            currentCharacter = "";
        }

        // Small delay before next frame
        await new Promise(
            resolve => setTimeout(resolve, 200)
        );

        if (running) {
            processFrame();
        }
    }

    processFrame();

})();
"""))

<IPython.core.display.Javascript object>

In [7]:
recognized_text = []
last_accepted_sign = None

total_recognized = 0
total_unknown = 0

def add_sign_to_text(sign):
    global last_accepted_sign, total_recognized

    # Ignore empty/unknown predictions
    if sign is None or sign == "Unknown":
        return

    # Prevent the same held sign from being added repeatedly
    if sign != last_accepted_sign:
        recognized_text.append(sign)
        last_accepted_sign = sign
        total_recognized += 1

def release_sign():
    """
    Call this when no valid sign is detected.
    This allows the same sign to be entered again later.
    """
    global last_accepted_sign
    last_accepted_sign = None

def add_space():
    recognized_text.append(" ")

def delete_last():
    if recognized_text:
        recognized_text.pop()

def clear_text():
    global recognized_text, last_accepted_sign
    recognized_text = []
    last_accepted_sign = None

def get_text():
    return " ".join(
        item for item in recognized_text
        if item != " "
    )

print("✓ Text memory initialized")
print("✓ Duplicate-sign prevention initialized")
print("✓ Session statistics initialized")

✓ Text memory initialized
✓ Duplicate-sign prevention initialized
✓ Session statistics initialized


In [25]:
import gradio as gr
import numpy as np
import mediapipe as mp
from collections import Counter, deque

gradio_prediction_history = deque(maxlen=7)

def gradio_recognize(frame):

    global gradio_prediction_history

    if frame is None:
        gradio_prediction_history.clear()
        return "", "Waiting for camera...", "—"

    try:

        # Gradio webcam provides RGB NumPy image
        frame = np.asarray(frame, dtype=np.uint8)

        # Create MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame
        )

        # Detect hand
        result = detector.detect(mp_image)

        if not result.hand_landmarks:

            gradio_prediction_history.clear()

            return "", "No hand", "—"

        landmarks_63 = normalize_landmarks(
            result.hand_landmarks[0]
        )

        features_28 = make_geometry_only_features(
            landmarks_63
        )

        probabilities = classifier.predict_proba(
            [features_28]
        )[0]

        best_index = np.argmax(probabilities)

        raw_prediction = str(
            classifier.classes_[best_index]
        )

        confidence = float(
            probabilities[best_index]
        )

        # UNKNOWN REJECTION

        if confidence < CONFIDENCE_THRESHOLD:
            raw_prediction = "Unknown"

        # TEMPORAL SMOOTHING

        gradio_prediction_history.append(
            raw_prediction
        )

        # Need several frames before accepting prediction
        if len(gradio_prediction_history) < 5:

            return (
                "",
                "Reading...",
                f"{confidence * 100:.1f}%"
            )

        counts = Counter(
            gradio_prediction_history
        )

        stable_label, count = (
            counts.most_common(1)[0]
        )

        # Require 5 matching predictions
        if count < 5:

            return (
                "",
                "Reading...",
                f"{confidence * 100:.1f}%"
            )

        if stable_label == "Unknown":

            return (
                "",
                "Unknown",
                f"{confidence * 100:.1f}%"
            )

        nepali, english = DISPLAY_NAMES.get(
            stable_label,
            ("", stable_label)
        )

        return (
            nepali,
            english,
            f"{confidence * 100:.1f}%"
        )

    except Exception as e:

        return (
            "",
            "Error",
            str(e)
        )

def add_character(character, text):

    text = text or ""

    if character and character.strip():
        return text + character

    return text

def add_space(text):

    return (text or "") + " "

def delete_character(text):

    text = text or ""

    if not text:
        return ""

    return text[:-1]

def clear_text():

    return ""

custom_css = """

/* =========================================================
   APPLICATION
========================================================= */

.gradio-container {
    max-width: 1080px !important;
    margin: 0 auto !important;
    padding-top: 5px !important;
}

/* =========================================================
   HEADER
========================================================= */

#project-header {
    background: #245c3a !important;
    border-bottom: 4px solid #e3b505 !important;
    border-radius: 7px !important;

    padding: 7px 16px !important;
    margin: 0 0 5px 0 !important;
}

#project-header h1 {
    color: #ffffff !important;

    font-size: 21px !important;
    line-height: 1.15 !important;

    margin: 0 !important;
    padding: 0 !important;
}

#project-header p {
    color: #f5d547 !important;

    font-size: 12px !important;

    margin: 3px 0 0 0 !important;
    padding: 0 !important;
}

/* =========================================================
   TABS
========================================================= */

.tab-nav {
    margin-top: 0 !important;
    margin-bottom: 4px !important;
}

.tab-nav button {
    font-size: 14px !important;
    font-weight: 600 !important;
}

/* =========================================================
   RECOGNITION INSTRUCTION
========================================================= */

#instruction-box {
    background: #fff8d6 !important;
    color: #202020 !important;

    border: 1px solid #e3b505 !important;
    border-radius: 6px !important;

    padding: 5px 10px !important;
    margin: 0 0 5px 0 !important;

    font-size: 12px !important;
}

#instruction-box * {
    color: #202020 !important;
}

/* =========================================================
   CAMERA
========================================================= */

#camera-box {
    margin: 0 !important;
}

/* =========================================================
   PREDICTION PANEL
========================================================= */

#prediction-card {
    background: #fff8d6 !important;

    border: 1px solid #e3b505 !important;
    border-radius: 7px !important;

    padding: 8px !important;
    margin: 0 !important;
}

/* Labels inside prediction panel */

#prediction-card label,
#prediction-card span {
    color: #ffffff !important;
}

/* Nepali character */

#nepali-output textarea {
    color: #245c3a !important;

    font-size: 38px !important;
    font-weight: bold !important;
    text-align: center !important;

    min-height: 52px !important;
}

/* Other result fields */

#prediction-card textarea {
    min-height: 37px !important;
}

/* =========================================================
   TEXT BUILDER
========================================================= */

#text-builder {
    margin-top: 2px !important;
}

#text-builder textarea {
    font-size: 21px !important;
    min-height: 45px !important;
}

/* =========================================================
   BUTTONS
========================================================= */

.green-button {
    background: #245c3a !important;
    color: #ffffff !important;

    border: 1px solid #245c3a !important;

    font-weight: 600 !important;
}

.green-button:hover {
    background: #1d4d30 !important;
}

.yellow-button {
    background: #e3b505 !important;
    color: #202020 !important;

    border: 1px solid #e3b505 !important;

    font-weight: 600 !important;
}

.yellow-button:hover {
    background: #cda304 !important;
}

/* =========================================================
   SECTION HEADINGS
========================================================= */

.section-title h2 {
    color: #45a568 !important;
}

.section-title h3 {
    color: #45a568 !important;
}

/* =========================================================
   ABOUT INTRODUCTION
========================================================= */

.info-card {
    background: #fff8d6 !important;
    color: #202020 !important;

    border: 1px solid #e3b505 !important;
    border-radius: 7px !important;

    padding: 14px 18px !important;
    margin-bottom: 10px !important;

    line-height: 1.6 !important;
}

.info-card,
.info-card p,
.info-card b,
.info-card strong,
.info-card span {
    color: #202020 !important;
}

/* =========================================================
   DARK GREEN INFORMATION CARDS
========================================================= */

.green-card {
    background: #10281c !important;
    color: #eeeeee !important;

    /* ONLY ONE BORDER */
    border: 1px solid #245c3a !important;
    border-radius: 7px !important;

    padding: 14px 18px !important;
}

/* Everything inside green cards */

.green-card p,
.green-card li,
.green-card strong,
.green-card span {
    color: #eeeeee !important;
}

/* Headings inside green cards */

.green-card h3 {
    color: #49a96d !important;
}

/* =========================================================
   PIPELINE
========================================================= */

.pipeline {
    background: #fff8d6 !important;
    color: #202020 !important;

    border: 1px solid #e3b505 !important;
    border-radius: 7px !important;

    padding: 13px 15px !important;

    text-align: center !important;

    font-size: 15px !important;
    font-weight: 600 !important;
    line-height: 1.6 !important;
}

.pipeline,
.pipeline p,
.pipeline strong,
.pipeline span {
    color: #202020 !important;
}

/* =========================================================
   SPACING
========================================================= */

.gradio-container .gap {
    gap: 7px !important;
}

"""

with gr.Blocks(
    css=custom_css,
    title="Sign Language Recognition System"
) as final_nsl_app:

    gr.Markdown(
        """
# Sign Language Recognition System
Nepali Sign Language Recognition Using Computer Vision and Machine Learning
        """,
        elem_id="project-header"
    )

    with gr.Tabs():

        with gr.Tab("Recognition"):

            gr.Markdown(
                """
**Live Recognition:** Position your hand clearly in front of the camera and hold the sign briefly for a stable prediction.
                """,
                elem_id="instruction-box"
            )

            with gr.Row(equal_height=True):

                with gr.Column(scale=3):

                    camera = gr.Image(
                        sources=["webcam"],
                        type="numpy",
                        label="Live Camera",
                        streaming=True,
                        height=280,
                        elem_id="camera-box"
                    )

                with gr.Column(
                    scale=2,
                    elem_id="prediction-card"
                ):

                    nepali_output = gr.Textbox(
                        label="Nepali Character",
                        interactive=False,
                        elem_id="nepali-output"
                    )

                    english_output = gr.Textbox(
                        label="Recognized Sign",
                        value="Waiting for camera...",
                        interactive=False
                    )

                    confidence_output = gr.Textbox(
                        label="Confidence",
                        value="—",
                        interactive=False
                    )

            camera.stream(
                fn=gradio_recognize,
                inputs=camera,
                outputs=[
                    nepali_output,
                    english_output,
                    confidence_output
                ],
                time_limit=120,
                stream_every=0.25
            )

            built_text = gr.Textbox(
                label="Built Text",
                placeholder="Add recognized characters here...",
                lines=1,
                elem_id="text-builder"
            )

            with gr.Row():

                add_button = gr.Button(
                    "＋ Add Character",
                    elem_classes="green-button"
                )

                space_button = gr.Button(
                    "Space"
                )

                delete_button = gr.Button(
                    "Delete"
                )

                clear_button = gr.Button(
                    "Clear",
                    elem_classes="yellow-button"
                )

            add_button.click(
                fn=add_character,
                inputs=[
                    nepali_output,
                    built_text
                ],
                outputs=built_text
            )

            space_button.click(
                fn=add_space,
                inputs=built_text,
                outputs=built_text
            )

            delete_button.click(
                fn=delete_character,
                inputs=built_text,
                outputs=built_text
            )

            clear_button.click(
                fn=clear_text,
                inputs=None,
                outputs=built_text
            )

        with gr.Tab("About"):

            gr.Markdown(
                """
                """,
                elem_classes="section-title"
            )

            gr.HTML(
                """
<div class="info-card">

<strong>Sign Language Recognition System</strong> is a computer
vision and machine-learning project developed to recognize
Nepali Sign Language hand gestures.

<br><br>

The system captures hand gestures using a camera, detects
important hand landmarks and processes geometric information
from the detected hand.

<br><br>

A trained Random Forest classifier analyzes these features
and predicts the corresponding Nepali Sign Language character.

<br><br>

Recognized characters can also be combined into text using
the built-in text builder.

</div>
                """
            )

            with gr.Row():

                with gr.Column():

                    gr.HTML(
                        """
<div class="green-card">

<h3>Main Features</h3>

<ul>
    <li>Real-time hand detection</li>
    <li>21 hand landmark extraction</li>
    <li>Landmark normalization</li>
    <li>Geometric feature extraction</li>
    <li>Random Forest classification</li>
    <li>Prediction confidence</li>
    <li>Unknown-sign rejection</li>
    <li>Temporal prediction smoothing</li>
    <li>Nepali character display</li>
    <li>Text construction</li>
</ul>

</div>
                        """
                    )

                with gr.Column():

                    gr.HTML(
                        """
<div class="green-card">

<h3>Technologies Used</h3>

<ul>
    <li>Python</li>
    <li>MediaPipe</li>
    <li>OpenCV</li>
    <li>Scikit-learn</li>
    <li>Random Forest</li>
    <li>Gradio</li>
    <li>Google Colab</li>
</ul>

<h3>Project Type</h3>

<p>
Bachelor of Computer Application<br>
Final Project
</p>

</div>
                        """
                    )

        with gr.Tab("How It Works"):

            gr.Markdown(
                """
## How the System Works
                """,
                elem_classes="section-title"
            )

            with gr.Row():

                with gr.Column():

                    gr.HTML(
                        """
<div class="green-card">

<h3>1. Camera Input</h3>

<p>
The webcam captures the user's hand gesture.
</p>

<h3>2. Hand Detection</h3>

<p>
MediaPipe detects the hand and extracts
<strong>21 hand landmarks</strong>.
</p>

<h3>3. Landmark Normalization</h3>

<p>
The landmark coordinates are normalized relative to
the hand so recognition is less dependent on its
position and size.
</p>

<h3>4. Feature Extraction</h3>

<p>
Geometric information such as joint angles and
distances between fingertips is calculated from
the normalized landmarks.
</p>

</div>
                        """
                    )

                with gr.Column():

                    gr.HTML(
                        """
<div class="green-card">

<h3>5. Sign Classification</h3>

<p>
The extracted geometric features are supplied to
the trained <strong>Random Forest classifier</strong>.
</p>

<h3>6. Confidence Check</h3>

<p>
Predictions below the required confidence threshold
can be treated as <strong>Unknown</strong> instead
of forcing a sign prediction.
</p>

<h3>7. Temporal Smoothing</h3>

<p>
Predictions from multiple camera frames are considered
before displaying a stable result.
</p>

<h3>8. Text Construction</h3>

<p>
The recognized Nepali character can be added to the
text builder. The user can also add spaces, delete
characters or clear the text.
</p>

</div>
                        """
                    )

            gr.HTML(
                """
<div class="pipeline">

Camera → MediaPipe → 21 Hand Landmarks →
Normalization → Geometric Features →
Random Forest → Confidence Check →
Temporal Smoothing → Prediction → Text

</div>
                """
            )

print("✓ Sign Language Recognition System ready")
print("✓ Recognition ready")
print("✓ About ready")
print("✓ How It Works ready")

final_nsl_app.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_621/1578193051.py:511: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✓ Sign Language Recognition System ready
✓ Recognition ready
✓ About ready
✓ How It Works ready
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17b216f2740a116b19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
